### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="credit_card_clients_default",
    dataset_year="2009",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C55S3H",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/credit_card_clients_default/ && wget -P local-data-warehouse/credit_card_clients_default/ https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip && unzip local-data-warehouse/credit_card_clients_default/default+of+credit+card+clients.zip -d local-data-warehouse/credit_card_clients_default/ && rm local-data-warehouse/credit_card_clients_default/default+of+credit+card+clients.zip
""",
    # References
    academic_reference_bibtex="""@article{Yeh2009TheCD,
  title={The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients},
  author={I-Cheng Yeh and Che-hui Lien},
  journal={Expert Systems with Applications},
  year={2009},
  volume={36},
  number={2},
  pages={2473-2480},
  doi={10.1016/j.eswa.2007.12.020},
  url={https://doi.org/10.1016/j.eswa.2007.12.020}
}
""",
    academic_reference_bibtex_key="Yeh2009TheCD",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
  - We rename the target variable and restore the original class names.
  - We drop the "ID" column.
  - Anomaly: the data has temporal features but the task is time-invariant.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DefaultOnPaymentNextMonth",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
)

## Preprocessing

In [69]:
import pandas as pd

df = pd.read_excel(dataset_mold.path / "default of credit card clients.xls")

df.columns = df.iloc[0].str.lower()
df = df.iloc[1:].reset_index(drop=True)
df = df.drop(columns=["id"])

target_feature = "DefaultOnPaymentNextMonth"
df = df.rename(columns={"default payment next month": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"}).astype("category")

cat_columns = ["sex", "marriage", "education"]
df[cat_columns] = df[cat_columns].astype("category")

non_cat_cols = df.columns[~df.dtypes.eq("category")]
df[non_cat_cols] = df[non_cat_cols].apply(pd.to_numeric, errors="coerce") #returns NaN if conversion to numeric not possible

print("Loaded data shape:", df.shape)

Loaded data shape: (30000, 24)


In [70]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,DefaultOnPaymentNextMonth
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,Yes
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,Yes
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,No
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,No
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,No


## Data Checks

In [71]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 30,000
Columns: 24
Use sampling: False (sample size: 30,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt6']
Rows remaining as candidates after top-10 filter: 1,248 (of 30,000)

#### Duplicate Report
Total duplicate rows: 35 (0.12% of dataset)
Duplicate rows ignoring target: 56 (0.19% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [73]:
# Sample Rows
df_head

,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,DefaultOnPaymentNextMonth
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,Yes
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,Yes
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,No
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,No
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,No


In [74]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,sex,category,0.0,0.0,2.0,"2, 1"
1,education,category,0.0,0.0,7.0,"2, 1, 3, 5, 4, 6, 0"
2,marriage,category,0.0,0.0,4.0,"2, 1, 3, 0"
3,DefaultOnPaymentNextMonth,category,0.0,0.0,2.0,"No, Yes"
4,limit_bal,int64,0.0,0.0,81.0,"50000, 20000, 30000, 80000, 200000, 150000, 100000, 180000, 360000, 60000"
5,age,int64,0.0,0.0,56.0,"29, 27, 28, 30, 26, 31, 25, 34, 32, 33"
6,pay_0,int64,0.0,0.0,11.0,"0, -1, 1, -2, 2, 3, 4, 5, 8, 6"
7,pay_2,int64,0.0,0.0,11.0,"0, -1, 2, -2, 3, 4, 1, 5, 7, 6"
8,pay_3,int64,0.0,0.0,11.0,"0, -1, -2, 2, 3, 4, 7, 6, 5, 1"
9,pay_4,int64,0.0,0.0,11.0,"0, -1, -2, 2, 3, 4, 7, 5, 6, 1"


In [75]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
limit_bal,30000.0,167484.322667,129747.661567,10000.0,1000000.0
age,30000.0,35.485500,9.217904,21.0,79.0
pay_0,30000.0,-0.016700,1.123802,-2.0,8.0
pay_2,30000.0,-0.133767,1.197186,-2.0,8.0
pay_3,30000.0,-0.166200,1.196868,-2.0,8.0
pay_4,30000.0,-0.220667,1.169139,-2.0,8.0
pay_5,30000.0,-0.266200,1.133187,-2.0,8.0
pay_6,30000.0,-0.291100,1.149988,-2.0,8.0
bill_amt1,30000.0,51223.330900,73635.860576,-165580.0,964511.0
bill_amt2,30000.0,49179.075167,71173.768783,-69777.0,983931.0


In [76]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                    rank                    
DefaultOnPaymentNextMonth 1       No  23364  77.88
                          2      Yes   6636  22.12
education                 1        2  14030  46.77
                          2        1  10585  35.28
                          3        3   4917  16.39
                          4        5    280   0.93
                          5        4    123   0.41
marriage                  1        2  15964  53.21
                          2        1  13659  45.53
                          3        3    323   1.08
                          4        0     54   0.18
sex                       1        2  18112  60.37
                          2        1  11888  39.63

In [77]:
# Target Distribution
target_df

,count,pct
DefaultOnPaymentNextMonth,,
No,23364,77.88
Yes,6636,22.12


## Task Curation

In [78]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [79]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

# The structure of splits is:
# splits = {
#     repeat_i: {
#         fold_i: (train_idx, test_idx),
#     }
# }

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [80]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cbee8-4ec7-7749-8f9b-c4109d0d4b22
1959bdc483071e898ffdc444bb71d18259d2a8c8925774278301292ac7b1722a
